In [1]:
import json
#import torch
from transformers import BertTokenizer, TFBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tqdm import tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
data_dir = '/content/drive/MyDrive/Healthcare_Sentiment/CNBC/train_items.jl'  # Replace with your actual path

In [ ]:
# Load data from JSON Lines file
#def load_data(file_path):
#    texts = []
#    labels = []
#    with open(file_path, 'r') as f:
#        for line in f:
#            st = r'%s' % line
#            data = json.loads(st)
#            texts.append(data['text'])
#            labels.append(data['label'])
#    return texts, labels

In [4]:
def load_data_with_cleaning(file_path):
        texts = []
        labels = []
        with open(file_path, 'r') as f:
            for line in f:
                try:
                    # Example cleaning: Replace literal backslashes with escaped ones
                    # This is a simplification and might not work for all issues!
                    cleaned_line = line.replace('\\"', '/')
                    #cleaned_line = cleaned_line1.replace('','')
                    # Ensure the cleaned line is a valid JSON string
                    data = json.loads(cleaned_line)
                    #combined_text = f"{data['title']} {data['text']}"
                    texts.append(data['title'])
                    #labels.append(data['label'])
                    # Convert label to integer
                    labels.append(int(data['label']))
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON on line: {e}")
                    print(f"Problematic line content (first 200 chars): {line[:1535]}")
                    # Decide how to handle errors (skip line, raise error, etc.)
                    continue # Skip the problematic line

        return texts, labels

In [5]:
texts, labels = load_data_with_cleaning(data_dir)

In [6]:
print(texts)

["UnitedHealth's guidance cut may mean trouble for some insurers", 'Brain implant cleared by FDA for Precision Neuroscience, a Musk Neuralink rival', 'Healthy Returns: Trump seeks to change Medicare drug price negotiations', 'Merck lowers profit outlook, partly due to $200 million expected tariff hit', 'Bristol Myers Squibb tops estimates, hikes outlook as it braces for tariffs', "Why we're lowering our Bristol Myers price target despite an earnings beat", 'Novo Nordisk scores major legal win that bars many Wegovy, Ozempic copies', 'Oracle engineers caused dayslong software outage at U.S. hospitals', 'Novo Nordisk opens Wegovy to telehealth in push for new patients', 'Pfizer CEO says tariff uncertainty is deterring further U.S. investment', 'GLP-1s can help employers lower medical costs in 2 years, new study finds', 'CVS tops estimates, hikes guidance as insurance business shows some improvement', "CVS to boost access to Novo Nordisk's Wegovy for patients on its drug plans", "From 'Coc

In [ ]:
#load_data(data_dir)

In [7]:
#Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)


In [8]:
print(f"Number of training samples: {len(train_texts)}")

Number of training samples: 31


In [23]:
# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(set(labels)))

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [24]:
# Tokenize data
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=100)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=100)

In [25]:
# Convert to TensorFlow datasets
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_labels
)).batch(4)

In [26]:
val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings),
    val_labels
)).batch(4)

In [27]:
# Optimizer and loss
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')

In [28]:
# Compile model
model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

In [29]:
# Train model
model.fit(train_dataset, epochs=5, validation_data=val_dataset
    )

Epoch 1/5
8/8 [==============================] - 50s 645ms/step - loss: 0.7377 - accuracy: 0.5806 - val_loss: 0.6838 - val_accuracy: 0.3750
Epoch 2/5
8/8 [==============================] - 1s 105ms/step - loss: 0.5271 - accuracy: 0.8065 - val_loss: 0.4958 - val_accuracy: 1.0000
Epoch 3/5
8/8 [==============================] - 1s 77ms/step - loss: 0.3101 - accuracy: 0.9355 - val_loss: 0.3718 - val_accuracy: 0.7500
Epoch 4/5
8/8 [==============================] - 1s 70ms/step - loss: 0.1925 - accuracy: 0.9677 - val_loss: 0.1794 - val_accuracy: 1.0000
Epoch 5/5
8/8 [==============================] - 1s 71ms/step - loss: 0.1072 - accuracy: 0.9677 - val_loss: 0.4485 - val_accuracy: 0.7500


In [30]:
# Evaluate model
loss, accuracy = model.evaluate(val_dataset)
print(f"Loss: {loss}, Accuracy: {accuracy}")

2/2 [==============================] - 0s 27ms/step - loss: 0.4485 - accuracy: 0.7500
Loss: 0.4484584927558899, Accuracy: 0.75


In [31]:
# Make predictions
text = "This movie was great!"
predict_input = tokenizer(text, truncation=True, padding=True, return_tensors='tf')
output = model(predict_input)[0]
prediction_value = tf.argmax(output, axis=1).numpy()[0]
print(f"Predicted sentiment: {prediction_value}")

Predicted sentiment: 0


In [32]:
#File to Apply model to
new_data_dir = '/content/drive/MyDrive/Healthcare_Sentiment/CNBC/items.jl'  # Replace with your actual path

In [33]:
# Function to load data from a JSON Lines file (similar to your existing function)
def load_new_data_with_cleaning(file_path):
    contents = []
    texts = []
    ids = [] # Assuming each item has a unique ID you want to keep


    source = []
    not_text = []
    author = []
    timestamp = []
    tags = []
    summary = []
    related_articles = []
    metadata = []
    category = []


    with open(file_path, 'r') as f:
        for line in f:
            try:
                cleaned_line = line.replace('\\"', '/')
                data = json.loads(cleaned_line)
                # Assuming you want to predict on the 'title' again
                texts.append(data['title'])
                contents.append(data['content'])


                source.append(data['source'])
                not_text.append(data['text'])
                author.append(data['author'])
                timestamp.append(data['timestamp'])
                tags.append(data['tags'])
                summary.append(data['summary'])
                related_articles.append(data['related_articles'])
                metadata.append(data['metadata'])
                category.append(data['category'])


                # Assuming an 'id' field exists to track the original item
                if 'url' in data:
                  ids.append(data['url'])
                else:
                  ids.append(None) # Or handle cases without an ID
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON on line: {e}")
                print(f"Problematic line content (first 200 chars): {line[:1535]}")
                continue
    return contents, texts, ids, source, not_text, author, timestamp, tags, summary, related_articles, metadata, category

# Load the new data
new_content, new_texts, new_ids, new_source, new_not_text, new_author, new_timestamp, new_tags, new_summary, new_related_articles, new_metadata, new_category = load_new_data_with_cleaning(new_data_dir)

# Tokenize the new data
new_encodings = tokenizer(list(new_texts), truncation=True, padding=True, max_length=100, return_tensors='tf')

# Create a TensorFlow dataset for the new data
new_dataset = tf.data.Dataset.from_tensor_slices(
    dict(new_encodings)
).batch(4)

# Make predictions
predictions = model.predict(new_dataset)

# Get the predicted class (index with the highest probability)
predicted_labels = tf.argmax(predictions.logits, axis=1).numpy()

# Write the results back to a new file or overwrite the original
output_data_dir = '/content/drive/MyDrive/Healthcare_Sentiment/CNBC/new_items_with_predictions.jl' # Define output file path

with open(output_data_dir, 'w') as outfile:
    for i, text in enumerate(new_texts):
        result = {
            'url': new_ids[i], # Include the original ID if available

            'source': new_source[i],

            'title': new_texts[i],
            'text': new_not_text[i],

            'author': new_author[i],
            'timestamp': new_timestamp[i],
            'tags': new_tags[i],
            'content': new_content[i],
            'summary': new_summary[i],
            'related_articles': new_related_articles[i],
            'metadata': new_metadata[i],
            'category': new_category[i],

            'predicted_label': int(predicted_labels[i]) # Ensure it's an integer
        }
        outfile.write(json.dumps(result) + '\n')

print(f"Predictions written to {output_data_dir}")

49/49 [==============================] - 5s 30ms/step
Predictions written to /content/drive/MyDrive/Healthcare_Sentiment/CNBC/new_items_with_predictions.jl
